# Fraudulent Transaction Detection for Digital Money Transfer

## Introduction

Novapay is a digital-first money transfer company headquartered in Toronto, Canada, with operations across the UK, Canada, and United States.

**Problem** 

Threats from fraudulent activities including account takeovers, identity theft, and unauthorised transactions.

**Existing System**
Relies on static thresholds and manual reviews. False positives frustrate customers while missed fraud incidents exposes the business financial and compliance risks.

**Task**

Design and deploy a machine learning driven fraud detection system that can accurately classify transactions as fraudulent or legitimate, adapt to changing patterns, and provide transparent, explainable decisions.


## Import Necessary Libraries

In [365]:
# Import libraries
import pandas as pd
import numpy as np

# Data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Machine learning libraries
from sklearn.model_selection import train_test_split

## Data Collection & Profiling

- **Load the dataset**:

Load and display the first five rows to see what data is recorded.

In [366]:
# Load the dataset
df = pd.read_csv(r'C:\Users\ugwuc\OneDrive\Desktop\Amdari_Projects\Fraud_detection\nova_pay_combined.csv')
df.head()   # Display the first five rows of the dataset

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


- **Shape and Data Type**: 

Check the shape and data type of the dataset

In [367]:
# Check for the shape of the dataset
print (f"Dataset shape: {df.shape}")

# Check for the data types of the columns
df.info()

Dataset shape: (11400, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address         

- **Missing values**: 

Check for any missing values in each column, and what percentage of the dataset is missing.

In [368]:
# Check for missing values
print(df.isnull().sum())

# What percentage of the dataset is missing?
total_missing = (df.isnull().sum()).sum()
percent_missing = total_missing / len(df) * 100
print("\nPercentage of missing values:\n", percent_missing)

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

Percentage of missing values:
 16.05263157894737


- **Duplicated Rows**:

The total number of duplicated values contained in the dataset

In [369]:
# Check for duplicate values
duplicates = df.duplicated().sum()
print("Total number of duplicated rows:",duplicates)


Total number of duplicated rows: 200


- **The Target Variable**:

This is a classification problem therefore our target variable is - **is_fraud**. Now we need to check the distribution of our target variable - how many were legitmate or fraud, and what is the percentage of each distribution.

Label description:
- **0**: Flagged as **Not fraudulent**

- **1**: Flagged as **Fraudulent**

In [370]:
# How many were fraudulent and how many were not?
target_counts = df['is_fraud'].value_counts()
print("Total number of transactions by fraud status:\n", target_counts)

# What is the percentage distribution of the target variable?
target_counts = df['is_fraud'].value_counts(normalize=True)
print("Percentage distribution of the target variable:\n", target_counts)

Total number of transactions by fraud status:
 is_fraud
0    10403
1      997
Name: count, dtype: int64
Percentage distribution of the target variable:
 is_fraud
0    0.912544
1    0.087456
Name: proportion, dtype: float64


- **Categorical Columns**

Data is categorical if it represents caharacteristics, labels, or groups that fit into distinct fixed categories.

In the dataset, check categorical columns with unique values contained in each

In [371]:
# Check for unique values in categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    unique_values = df[col].nunique()
    print(f"Column '{col}' has {unique_values} unique values.")

Column 'transaction_id' has 11200 unique values.
Column 'customer_id' has 1315 unique values.
Column 'timestamp' has 11141 unique values.
Column 'home_country' has 7 unique values.
Column 'source_currency' has 3 unique values.
Column 'dest_currency' has 9 unique values.
Column 'channel' has 12 unique values.
Column 'amount_src' has 9856 unique values.
Column 'device_id' has 2113 unique values.
Column 'ip_address' has 10900 unique values.
Column 'ip_country' has 9 unique values.
Column 'kyc_tier' has 14 unique values.


## Data Cleaning 

Here we'd address the issues noticed in the dataset such as - data type, missing values, duplicates, as well as potential outliers and imbalance 

### Stage 1

#### **Address Data Type**

There are two columns with incorrect data types which would be addressed here - timestamp, and amount_src column

In [372]:
# Change timestamp data type to datatime format
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Change amount_src data type to a float
df['amount_src'] = pd.to_numeric(df['amount_src'], errors='coerce')

# Check the data types again to confirm the change
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  object             
 1   customer_id                11400 non-null  object             
 2   timestamp                  11339 non-null  datetime64[ns, UTC]
 3   home_country               11400 non-null  object             
 4   source_currency            11400 non-null  object             
 5   dest_currency              11400 non-null  object             
 6   channel                    11400 non-null  object             
 7   amount_src                 11396 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  de

#### **Handle Duplicated Rows**

Drop the duplicated rows in the dataset

In [373]:
# Drop duplicate rows
df.drop_duplicates(inplace=True)

# Check if duplicates were dropped
duplicates_after = df.duplicated().sum()
print("Total number of duplicated rows after dropping duplicates:", duplicates_after)

Total number of duplicated rows after dropping duplicates: 0


#### **Handle Missing Values**

The missingness in the dataset is 16% therefore the data won't be dropped as that could affect the quality of our analysis. Instead the missing values would be handled following the associated data

1. To handle missing values in amount_usd:
- Find the exchange rates. Since no missing value in amount_src, groupby source currency, then divide amount_usd by amount_src, and calculate its mean for each currency. Save result as a dictionary for easy lookup.

In [374]:
# Calculate the exchange rate and save to a dictionary

group_usd = df[df['amount_usd'].notna()].groupby('source_currency')
exchange_rate = group_usd.agg({'amount_usd': lambda x: (x / df.loc[x.index, 'amount_src']).mean()})
exchange_rate_dict = exchange_rate['amount_usd'].to_dict()
print("Exchange rate dictionary:\n", exchange_rate_dict)

Exchange rate dictionary:
 {'CAD': 0.7226052713792916, 'GBP': 1.224313741881689, 'USD': 0.983814123482574}


- Here handle the missing values in amount_usd using amount_src and the exchange_rate.
If amount_usd exist, keep it, else fill by multiplying amount_src with exchange_rate of that source currency.

In [375]:
# Handle the missing values in amount_usd
df['amount_usd'] = df.apply(
    lambda row: row['amount_src'] * exchange_rate_dict.get(
        row['source_currency'], np.nan) if pd.isnull(
            row['amount_usd']) else row['amount_usd'],
    axis=1
)

2. Handle missing values in fee:

Since the fee is traceable to the channel of transaction,
- If channels exist, fill missing fee per channel by channel median
- Then fill the remaining missing fee with the overall median

In [376]:
# If channels exist, fill missing fee per channel by channel median
# Then fill the remaining missing fee with the overall median
if 'fee' in df.columns and 'channel' in df.columns:
    df['fee'] = df.groupby('channel')['fee'].transform(lambda x: x.fillna(x.median()))
    df['fee'] = df['fee'].fillna(df['fee'].median())

3. Handling missing values in ip_country
- Fill the missing values to fallback to the corresponding home_country

In [377]:
# fill missing values in ip_country to fallback to the corresponding home_country
if {'ip_country', 'home_country'}.issubset(df.columns):
    df['ip_country'] = df.apply(
        lambda row: row['home_country'] if pd.isnull(row['ip_country']) else row['ip_country'], axis=1
    )

4. Handle missing values in 'kyc_tier' column
- Fill with kyc_tier most frequent value - Mode, otherwise fill with standard as default

In [378]:
# Fill with kyc_tier most frequent value - Mode, otherwise default to standard
if 'kyc_tier' in df.columns:
    most_freq_kyc = df['kyc_tier'].mode()[0] if not df['kyc_tier'].mode().empty else 'standard'
    df['kyc_tier'] = df['kyc_tier'].fillna(most_freq_kyc)

5. Handle missing values in 'device_trust_score' column
- Fill with the median of the group (new_device, kyc_tier)

In [379]:
# Handle missing values in device_trust_score column
# fill with the median of the group - new_device, kyc_tier
if 'device_trust_score' in df.columns and {'new_device', 'kyc_tier'}.issubset(df.columns):
    df['device_trust_score'] = df.groupby(['new_device', 'kyc_tier'])['device_trust_score'].transform(
        lambda x: x.fillna(x.median())
    )
    df['device_trust_score'] = df['device_trust_score'].fillna(df['device_trust_score'].median())

C:\Users\ugwuc\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ugwuc\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


6. Handle other missing values in: 
- 'timestamp' and 'amount_src': These are values that could not be converted during the data type changes intially performed.
- ip_address: drop ip_address

Therefore drop all these NaN, NaT, and null values

In [380]:
# Drop null values in timestamp, amount_src
df.dropna(subset = ['timestamp', 'amount_src', 'ip_address'], inplace=True)

Confirming that there are no more missing values in the dataset

In [381]:
# print the number of missing values after handling them
print("\nMissing values after handling:\n", df.isnull().sum())


Missing values after handling:
 transaction_id               0
customer_id                  0
timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
device_id                    0
new_device                   0
ip_address                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
dtype: int64


### Stage 2

Here, I'd be checking the sanity and quality of our dataset and perform any further cleaning in other to prepare the dataset.

1. **Numerical Column (check)**

First, I will use the describe() to gain insight into the content of each numerical column

In [382]:
# Describe the dataset
df.describe(include='all')

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
count,10836,10836,10836,10836,10836,10836,10836,10836.000000,10836.000000,10836.000000,...,10836.000000,10836,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000
unique,10836,1314,NaN,7,3,9,12,NaN,NaN,NaN,...,NaN,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,fdffeb16-192a-4483-9b1e-9928e23269c2,402cccc9-28de-45b3-9af7-cc5302aa1f93,NaN,US,USD,NGN,mobile,NaN,NaN,NaN,...,NaN,standard,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,1433,NaN,7533,7619,1400,6016,NaN,NaN,NaN,...,NaN,7733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,2024-05-03 11:39:15.294875648+00:00,NaN,NaN,NaN,NaN,437.384918,448.258985,97.071335,...,0.398331,NaN,392.088778,0.654070,0.050941,0.268593,0.475914,0.749169,0.045484,0.090993
min,NaN,NaN,2022-10-03 18:40:59.468549+00:00,NaN,NaN,NaN,NaN,-9997.160000,7.230000,-1.000000,...,0.004000,NaN,1.000000,-0.100000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,2023-08-15 06:58:52.468549120+00:00,NaN,NaN,NaN,NaN,90.910000,92.600000,2.390000,...,0.209000,NaN,147.000000,0.515000,0.000000,0.169000,0.000000,0.000000,0.000000,0.000000
50%,NaN,NaN,2024-05-09 14:30:08.521080064+00:00,NaN,NaN,NaN,NaN,159.080000,163.590000,3.510000,...,0.326000,NaN,272.000000,0.658000,0.000000,0.223000,0.000000,0.000000,0.000000,0.000000
75%,NaN,NaN,2025-01-29 08:52:54.047345408+00:00,NaN,NaN,NaN,NaN,295.485000,302.682500,5.560000,...,0.489250,NaN,661.000000,0.894000,0.000000,0.391000,0.000000,0.000000,0.050000,0.000000
max,NaN,NaN,2025-12-16 00:13:41.468549+00:00,NaN,NaN,NaN,NaN,11942.890000,12497.900000,9999.990000,...,1.200000,NaN,1095.000000,0.999000,2.000000,0.900000,8.000000,11.000000,0.250000,1.000000


- **Address the negative values**: discover the count of these neagtive values and address with least as zero(0)

In [383]:
# Count the number of negatives in the numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
df[numerical_cols].apply(lambda x: (x < 0)).sum()


amount_src                    97
amount_usd                     0
fee                           89
exchange_rate_src_to_dest      0
ip_risk_score                  0
account_age_days               0
device_trust_score           186
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h              186
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [384]:
# For amount_src, fee, device_trust_score, and txn_velocity_1h
# Values should only be from 0 and above
df= df[
    (df['amount_src'] >= 0) &
    (df['fee'] >= 0) &
    (df['device_trust_score'] >= 0) &
    (df['txn_velocity_1h'] >= 0)
]


- **Correlations in numerical columns**: Where there any numerical columns that influence the other. Values closest to 1 shows a strong correlation

In [385]:
(df['amount_src'] / df['amount_usd']).describe()

count    10650.000000
mean         1.001101
std          0.144813
min          0.799741
25%          1.000000
50%          1.000000
75%          1.000000
max          1.351738
dtype: float64

2. **Categorical Columns (check)**

- Location mismatch

In [386]:
df['location_mismatch'].value_counts()

location_mismatch
False    8884
True     1766
Name: count, dtype: int64

Percentage of location mismatch

In [387]:
# percentage of location mismatch
location_mismatch_percentage = df['location_mismatch'].value_counts(normalize=True) * 100
print("\nPercentage of location mismatch:\n", location_mismatch_percentage)


Percentage of location mismatch:
 location_mismatch
False    83.41784
True     16.58216
Name: proportion, dtype: float64


Check if there are any inconsistencies in the categorical columns
- The value count of each unique values in the columns
- Address them

In [388]:
# List the categorical columns and their unique values
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    unique_values = df[col].nunique()
    print(f"Column '{col}' has {unique_values} unique values.")

Column 'transaction_id' has 10650 unique values.
Column 'customer_id' has 1313 unique values.
Column 'home_country' has 7 unique values.
Column 'source_currency' has 3 unique values.
Column 'dest_currency' has 9 unique values.
Column 'channel' has 12 unique values.
Column 'device_id' has 2110 unique values.
Column 'ip_address' has 10650 unique values.
Column 'ip_country' has 7 unique values.
Column 'kyc_tier' has 12 unique values.


**'home_country' Check**

In [389]:
# home_country unique values count
df['home_country'].unique()


array(['US', 'CA', 'UK', ' UK  ', ' US  ', 'unknown', ' CA  '],
      dtype=object)

Remove the whitespaces contained in the values

In [390]:
# Address the white space in the unique values of home_country
df['home_country'] = df['home_country'].str.strip()

#check again for unique values in home_country after stripping white space
df['home_country'].unique()

array(['US', 'CA', 'UK', 'unknown'], dtype=object)

Checked 'source_currency' and 'dest_currency' is fine

**Next check 'channel'**

In [391]:
# Check unique values in channel column
df['channel'].unique()

array(['ATM', 'web', 'mobile', 'WEB', ' web  ', 'MOBILE', 'unknown',
       'mobille', ' mobile  ', 'weeb', 'ATm', ' ATM  '], dtype=object)

Address the three issues with in this column:
- Whitespaces: remove
- Inconsistent Naming: change all channels to lower cases
- Spelling Error: replace the incorrect spellings

In [392]:
# Three things to be done here:
# 1. Address the white space
df['channel'] = df['channel'].str.strip()

# 2. Address the naming inconsistency to all upper case
df['channel'] = df['channel'].str.lower()

# 3. Spelling correction in one line of code
# MOBILLE should be MOBILE, and WEEB should be WEB
df['channel'] = df['channel'].str.replace(
    'mobille', 'mobile').str.replace(
        'weeb', 'web'
    )

# Check again
df['channel'].unique()

array(['atm', 'web', 'mobile', 'unknown'], dtype=object)

**'ip_country' Check**

In [393]:
# Unique values in ip_country
df['ip_country'].unique()

array(['US', 'CA', 'UK', ' US  ', 'unknown', ' CA  ', ' UK  '],
      dtype=object)

Remove the whitespaces in the values contained in ip_country column

In [394]:
# Address the whitespaces
df['ip_country'] = df['ip_country'].str.strip()

# Check again
df['ip_country'].unique()

array(['US', 'CA', 'UK', 'unknown'], dtype=object)

**'kyc_tier' check**

In [395]:
# kyc_tier unique values
df['kyc_tier'].unique()

array(['standard', 'enhanced', 'low', ' standard  ', 'standrd',
       ' enhanced  ', 'STANDARD', 'unknown', 'enhancd', ' low  ',
       'ENHANCED', 'LOW'], dtype=object)

Three inconsistencies to address:
- Incorrect Spelling: replace wrong spellings
- Whitespaces: remove whitespaces
- Case: maintain same case for all values (lower case)

In [396]:
# Remove whitespaces
# Maintain lower case for each value
# Replace wrong spellings with the correct ones
df['kyc_tier'] = df['kyc_tier'].str.strip().str.lower().str.replace(
    'standrd', 'standard').str.replace(
        'enhancd', 'enhanced'
    )

# Check again
df['kyc_tier'].unique()

array(['standard', 'enhanced', 'low', 'unknown'], dtype=object)

**Shape of the dataset after cleaning**

At this point our dataset is cleaned and ready for EDA

In [ ]:
# Check the shape of the dataset after cleaning
print(f"Dataset shape after cleaning: {df.shape}")

Dataset shape after cleaning: (10650, 26)


## Exploratory Data Analysis (EDA)